# 🏆 Minicurso Prático: Firestore (CRUD e Relacionamentos)
Este notebook contém a execução prática do nosso Banco de Dados NoSQL para a disciplina de IAAD. 
Abaixo, realizaremos a conexão com o Google Cloud e executaremos operações demonstrando as técnicas de **Aninhamento**, **Desnormalização** e **Referência**.

## 0. Configuração e Conexão

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore

print("Iniciando conexão com o Firestore...")
cred = credentials.Certificate("iaad-projeto-copa-do-mundo-firebase-adminsdk-fbsvc-ce1cc53869.json")

if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()
# Direcionamento específico para o banco de dados do projeto
db._database_string_internal = "projects/iaad-projeto-copa-do-mundo/databases/copa-do-mundo"

print("✅ Conexão estabelecida com sucesso!")

## 1. Operação CREATE (Armazenamento)
Nesta etapa, inserimos os dados aplicando nossa modelagem conceitual.
* **Seleção:** Jogadores são inseridos via *Aninhamento* (Embedding).
* **Partida:** O estádio é *Desnormalizado* e as seleções são salvas por *Referência* (ID).

In [ ]:
print("--- EXECUTANDO CREATE ---")

# 1. Inserindo Seleção (Aninhamento de Jogadores)
dados_brasil = {
    "nome_selecao": "Brasil",
    "continente": "América do Sul",
    "tecnico": "Dorival Júnior",
    "titulos": 5,
    "jogadores": [
        {"nome_jogador": "Vinícius Júnior", "posicao": "Atacante", "numero_camisa": 7},
        {"nome_jogador": "Alisson Becker", "posicao": "Goleiro", "numero_camisa": 1}
    ]
}
db.collection("selecoes").document("brasil").set(dados_brasil)
print("✅ Documento 'brasil' criado na coleção 'selecoes'.")

# 2. Inserindo Partida (Referência de Seleção e Desnormalização de Estádio)
dados_partida = {
    "data_partida": "2026-06-20",
    "selecao_1_id": "brasil",
    "selecao_2_id": "franca",
    "quantidade_gols_selecao_1": 2,
    "quantidade_gols_selecao_2": 1,
    "vencedor_id": "brasil",
    "estadio": {
        "nome_estadio": "Maracanã",
        "cidade": "Rio de Janeiro",
        "pais": "Brasil",
        "capacidade": 78838
    }
}
_, doc_ref = db.collection("partidas").add(dados_partida)
id_partida_criada = doc_ref.id  # Salvando para usar no Delete
print(f"✅ Partida criada! ID gerado automaticamente: {id_partida_criada}")

## 2. Operação READ (Consultas e Relacionamentos)
Aqui provamos como o NoSQL resolve os relacionamentos sem a cláusula `JOIN` do MySQL.

In [ ]:
print("\n--- CONSULTA 1: ANINHAMENTO (Acessando jogadores embutidos) ---")
doc_brasil = db.collection("selecoes").document("brasil").get()
if doc_brasil.exists:
    dados = doc_brasil.to_dict()
    print(f"Seleção: {dados['nome_selecao']}")
    print("Jogadores Convocados:")
    for jogador in dados.get("jogadores", []):
        print(f" - {jogador['numero_camisa']} | {jogador['nome_jogador']} ({jogador['posicao']})")

print("\n--- CONSULTA 2: DESNORMALIZAÇÃO (Filtrando por sub-documento) ---")
# Usando Notação de Ponto (Dot Notation) para achar estádios aninhados
query_maracana = db.collection("partidas").where("estadio.nome_estadio", "==", "Maracanã").stream()
for partida in query_maracana:
    dados_p = partida.to_dict()
    estadio = dados_p['estadio']
    print(f"Jogo em {dados_p['data_partida']} no {estadio['nome_estadio']} ({estadio['cidade']})")

print("\n--- CONSULTA 3: REFERÊNCIA (O 'JOIN' no código Python) ---")
partidas_ref = db.collection("partidas").limit(1).stream()
for partida in partidas_ref:
    dados_partida = partida.to_dict()
    id_vencedor = dados_partida['vencedor_id']
    
    # Segunda requisição simulando o comportamento de Chave Estrangeira
    doc_vencedor = db.collection("selecoes").document(id_vencedor).get()
    if doc_vencedor.exists:
        nome_vencedor = doc_vencedor.to_dict()['nome_selecao']
        print(f"🏆 O vencedor da partida foi a seleção do {nome_vencedor}!")

## 3. Operação UPDATE (Atualização)
Atualização de campos específicos e utilização de operadores atômicos do Firestore.

In [ ]:
print("--- EXECUTANDO UPDATE ---")

doc_ref = db.collection("selecoes").document("brasil")
doc_ref.update({
    "tecnico": "Novo Técnico Contratado",
    "titulos": firestore.Increment(1)  # Operador que soma +1 direto no banco
})

print("✅ Técnico atualizado e Títulos incrementados com sucesso!")

## 4. Operação DELETE (Remoção)
No Firestore, ao deletar um documento, todos os dados embutidos (como os jogadores) são excluídos simultaneamente.

In [ ]:
print("--- EXECUTANDO DELETE ---")

# Deletando a Seleção
db.collection("selecoes").document("brasil").delete()
print("✅ Seleção do Brasil (e seus jogadores) deletada com sucesso!")

# Deletando a Partida
try:
    db.collection("partidas").document(id_partida_criada).delete()
    print("✅ Partida de teste deletada com sucesso!")
except NameError:
    print("Atenção: A variável da partida não foi encontrada. Rode a célula CREATE primeiro.")